In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Preprocesamiento
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Selección de características
from sklearn.feature_selection import SequentialFeatureSelector

# Reducción de dimensionalidad
from sklearn.decomposition import PCA

# Model selection
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV

# Métricas
from sklearn.metrics import accuracy_score, classification_report, recall_score, roc_auc_score, roc_curve, auc, confusion_matrix, f1_score

# Pipelines y balanceo
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE


In [8]:
churn_train = pd.read_csv('churn_train_clean.csv')
churn_test = pd.read_csv('churn_test_clean.csv')

In [9]:
X_train = churn_train.drop(columns='Churn Value')
Y_train = churn_train['Churn Value']
X_test = churn_test.drop(columns='Churn Value')
Y_test = churn_test['Churn Value']

In [10]:
numerical_cols = ["Total Charges", "Monthly Charges", "Tenure Months", "CLTV"]
categorical_cols = [
    "Gender", "Senior Citizen", "Partner", "Dependents",
    "Phone Service", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method"
]


In [11]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])


In [ ]:

# Preprocesador
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# Pipelines
pipelines = {
    "Baseline": Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state=10, max_iter=1000))
    ]),
    "Under-sampling": Pipeline([
        ('preprocessor', preprocessor),
        ('undersample', RandomUnderSampler(random_state=10)),
        ('model', LogisticRegression(random_state=10, max_iter=1000))
    ]),
    "Over-sampling": Pipeline([
        ('preprocessor', preprocessor),
        ('oversample', RandomOverSampler(random_state=10)),
        ('model', LogisticRegression(random_state=10, max_iter=1000))
    ]),
    "SMOTE": Pipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=10)),
        ('model', LogisticRegression(random_state=10, max_iter=1000))
    ]),
    "Class-weight": Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state=10, max_iter=1000, class_weight='balanced'))
    ])
}

# Entrenar y evaluar
for name, pipe in pipelines.items():
    pipe.fit(X_train, Y_train)
    y_pred = pipe.predict(X_test)
    
    print(f"\n=== {name} ===")
    print(classification_report(Y_test, y_pred))



=== Baseline ===
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1294
           1       0.64      0.55      0.59       467

    accuracy                           0.80      1761
   macro avg       0.74      0.72      0.73      1761
weighted avg       0.79      0.80      0.79      1761



TypeError: All intermediate steps should be transformers and implement fit and transform or be the string 'passthrough' 'RandomUnderSampler(random_state=10)' (type <class 'imblearn.under_sampling._prototype_selection._random_under_sampler.RandomUnderSampler'>) doesn't